# 13. 운전자 "골든 루트" 발견

## 분석 배경 및 목적

Cramer & Krueger (2016)는 택시 운전자 간 효율성 차이가 상당하며, 상위 운전자들은 빈차 시간을 줄이고 실차율(utilization rate)을 높이는 전략적 OD 선택을 한다는 것을 보였다. 그러나 구체적으로 **어떤 OD 패턴이 고수익을 만드는지**는 밝히지 못했다.

본 분석은 이 질문에 답하기 위해 다음을 수행한다:

1. **상위/하위 10% 운전자 식별**: 총 매출 기준으로 운전자를 분류
2. **OD 패턴 비교**: 상위 운전자만 자주 이용하고 하위 운전자는 이용하지 않는 OD 쌍("골든 루트")을 식별
3. **골든 루트 특성 분석**: 평균 요금, 거리, 시간대 분포를 통해 왜 이 루트가 효율적인지 규명
4. **행동 패턴 비교**: 시간대별 운행 분포, 지역 선호도, 실차율 차이 분석

"골든 루트"란 단순히 요금이 높은 루트가 아니라, **빈차 시간 대비 매출이 극대화되는 루트**를 의미한다. 상위 운전자의 암묵지(tacit knowledge)를 데이터로 추출하여, 하위 운전자의 영업 전략 개선에 활용하는 것이 최종 목적이다.


In [ ]:
import gc, psutil, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'  # Mac: 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

CHUNK_SIZE = 1_000_000
D012_PATH = './DC_TBYXD012.csv'

DTYPE = {
    'PAY_AMT': 'int32',
    'RIDE_DIST': 'float32',
    'VACNTV_DIST': 'float32',
    'RIDE_A_CD': 'category',
    'ALIGHT_A_CD': 'category',
    'DRIVER_ID': 'category',
    'TAXI_VEHC_ID': 'category',
    'PLTF_FEE_AMT': 'int32',
}

def mem_usage():
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'RAM: {gb:.1f} GB')

mem_usage()

## 1. DRIVER_ID별 총 매출 집계 (chunk별)

운전자별 성과를 비교하기 위해 전체 기간의 총 매출을 집계한다. 매출 분포의 형태(정규, 편중 등)를 확인하고, 상위/하위 10% 경계값(quantile)을 산출한다. Cramer & Krueger (2016)의 방법론과 유사하게, 일정 기간 이상 운행한 운전자만을 분석 대상에 포함하여 표본 편향을 방지한다.


In [ ]:
driver_revenue = pd.Series(dtype='int64')  # DRIVER_ID -> 총 PAY_AMT

for i, chunk in enumerate(pd.read_csv(D012_PATH, chunksize=CHUNK_SIZE, dtype=DTYPE,
                                       usecols=['DRIVER_ID', 'PAY_AMT'])):
    agg = chunk.groupby('DRIVER_ID', observed=True)['PAY_AMT'].sum()
    driver_revenue = driver_revenue.add(agg, fill_value=0)
    if (i + 1) % 50 == 0:
        print(f'  chunk {i+1} done')
    del chunk, agg
    gc.collect()

driver_revenue = driver_revenue.astype('int64')
print(f'운전자 수: {len(driver_revenue):,}')
print(f'매출 통계:\n{driver_revenue.describe()}')
mem_usage()

In [ ]:
# 상위 10% / 하위 10% 기준선
q90 = driver_revenue.quantile(0.9)
q10 = driver_revenue.quantile(0.1)
print(f'상위 10% 기준 (>=): {q90:,.0f}원')
print(f'하위 10% 기준 (<=): {q10:,.0f}원')

top_drivers = set(driver_revenue[driver_revenue >= q90].index)
bot_drivers = set(driver_revenue[driver_revenue <= q10].index)
print(f'상위 10% 운전자: {len(top_drivers):,}명')
print(f'하위 10% 운전자: {len(bot_drivers):,}명')

## 2-3. 상위/하위 운전자 OD 패턴 Top 20

상위 10%와 하위 10% 운전자 각각의 빈도 상위 OD 쌍을 추출한다. 두 그룹의 OD 패턴 차이가 곧 "효율적 영업 전략"의 단서가 된다. 상위 운전자가 선호하는 OD는 (1) 다음 승객을 빠르게 잡을 수 있는 위치에 도착하거나, (2) 요금 대비 운행 시간이 효율적인 특성을 가질 것으로 가설을 설정한다.


In [ ]:
top_od = pd.Series(dtype='int64')  # (RIDE_A_CD, ALIGHT_A_CD) -> count
bot_od = pd.Series(dtype='int64')

cols = ['DRIVER_ID', 'RIDE_A_CD', 'ALIGHT_A_CD']

for i, chunk in enumerate(pd.read_csv(D012_PATH, chunksize=CHUNK_SIZE, dtype=DTYPE,
                                       usecols=cols)):
    chunk['DRIVER_ID'] = chunk['DRIVER_ID'].astype(str)
    chunk['OD'] = chunk['RIDE_A_CD'].astype(str) + ' -> ' + chunk['ALIGHT_A_CD'].astype(str)

    # 상위
    mask_top = chunk['DRIVER_ID'].isin(top_drivers)
    if mask_top.any():
        agg = chunk.loc[mask_top, 'OD'].value_counts()
        top_od = top_od.add(agg, fill_value=0)
    # 하위
    mask_bot = chunk['DRIVER_ID'].isin(bot_drivers)
    if mask_bot.any():
        agg = chunk.loc[mask_bot, 'OD'].value_counts()
        bot_od = bot_od.add(agg, fill_value=0)

    if (i + 1) % 50 == 0:
        print(f'  chunk {i+1} done')
    del chunk, mask_top, mask_bot
    gc.collect()

top_od = top_od.astype('int64').sort_values(ascending=False)
bot_od = bot_od.astype('int64').sort_values(ascending=False)
mem_usage()

In [ ]:
print('=== 상위 10% 운전자 OD 패턴 Top 20 ===')
print(top_od.head(20).to_frame('운행건수').to_string())
print()
print('=== 하위 10% 운전자 OD 패턴 Top 20 ===')
print(bot_od.head(20).to_frame('운행건수').to_string())

## 4. 골든 루트 식별

상위 10% 운전자의 빈도 Top OD 중 하위 10%에는 나타나지 않는 루트를 "골든 루트"로 정의한다. 이 차집합(set difference) 접근은 단순하지만 강력한데, 상위 운전자만의 **독점적 영업 패턴**을 직접적으로 포착하기 때문이다.


In [ ]:
# 상위 빈도 OD 비중으로 정규화
top_od_pct = (top_od / top_od.sum() * 100).rename('top_pct')
bot_od_pct = (bot_od / bot_od.sum() * 100).rename('bot_pct')

od_compare = pd.concat([top_od_pct, bot_od_pct], axis=1).fillna(0)
od_compare['gap'] = od_compare['top_pct'] - od_compare['bot_pct']

# 골든 루트: 상위 Top 100에 있으면서 하위 Top 100에 없는 루트
top_set = set(top_od.head(100).index)
bot_set = set(bot_od.head(100).index)
golden_routes = top_set - bot_set

golden_df = od_compare.loc[od_compare.index.isin(golden_routes)].sort_values('top_pct', ascending=False)
print(f'골든 루트 수: {len(golden_routes)}')
print()
print(golden_df.head(20).to_string())

In [ ]:
# 골든 루트 시각화 (Top 15)
plot_data = golden_df.head(15)

fig, ax = plt.subplots(figsize=(14, 7))
y_pos = range(len(plot_data))
ax.barh(y_pos, plot_data['top_pct'], color='#2196F3', label='상위 10%')
ax.barh(y_pos, plot_data['bot_pct'], color='#FF9800', alpha=0.7, label='하위 10%')
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_data.index, fontsize=9)
ax.set_xlabel('운행 비중 (%)')
ax.set_title('골든 루트: 상위 운전자만의 고빈도 OD (Top 15)')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. 골든 루트 특성 분석

골든 루트의 평균 요금, 평균 거리, 시간대 분포를 분석하여 이 루트가 왜 효율적인지를 규명한다. 단순히 요금이 높은 것이 아니라, **시간당 매출(revenue per hour)**이 높은 루트가 진정한 골든 루트다. 하차 지점의 다음 승객 확보 용이성(지역의 수요 밀도)도 중요한 요인이 된다.


In [ ]:
golden_route_list = list(golden_df.head(20).index)  # 분석 대상 골든 루트 Top 20

# chunk별로 골든 루트 운행 데이터 수집
golden_stats = []  # (OD, PAY_AMT, RIDE_DIST, hour)

cols = ['RIDE_DTIME', 'PAY_AMT', 'RIDE_DIST', 'RIDE_A_CD', 'ALIGHT_A_CD']

for i, chunk in enumerate(pd.read_csv(D012_PATH, chunksize=CHUNK_SIZE, dtype=DTYPE,
                                       usecols=cols)):
    chunk['OD'] = chunk['RIDE_A_CD'].astype(str) + ' -> ' + chunk['ALIGHT_A_CD'].astype(str)
    mask = chunk['OD'].isin(golden_route_list)
    if mask.any():
        sub = chunk.loc[mask, ['OD', 'PAY_AMT', 'RIDE_DIST', 'RIDE_DTIME']].copy()
        sub['hour'] = sub['RIDE_DTIME'].astype(str).str[8:10].astype('int8')
        golden_stats.append(sub[['OD', 'PAY_AMT', 'RIDE_DIST', 'hour']])
    if (i + 1) % 50 == 0:
        print(f'  chunk {i+1} done')
    del chunk
    gc.collect()

golden_all = pd.concat(golden_stats, ignore_index=True)
del golden_stats
gc.collect()
print(f'골든 루트 운행 건수: {len(golden_all):,}')
mem_usage()

In [ ]:
# 골든 루트별 평균 요금/거리
golden_summary = golden_all.groupby('OD').agg(
    건수=('PAY_AMT', 'count'),
    평균요금=('PAY_AMT', 'mean'),
    평균거리_m=('RIDE_DIST', 'mean')
).sort_values('건수', ascending=False)
golden_summary['평균요금'] = golden_summary['평균요금'].astype(int)
golden_summary['평균거리_m'] = golden_summary['평균거리_m'].astype(int)
print(golden_summary.head(20).to_string())

In [ ]:
# 골든 루트 시간대 분포
hour_dist = golden_all['hour'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(hour_dist.index, hour_dist.values, color='#2196F3', edgecolor='white')
ax.set_xlabel('시간대')
ax.set_ylabel('운행 건수')
ax.set_title('골든 루트 시간대별 분포')
ax.set_xticks(range(24))
plt.tight_layout()
plt.show()

del golden_all
gc.collect()

## 6. 상위 vs 하위 운전자 시간대별 운행 패턴

"잘 버는 기사는 몇 시에 일하는가"라는 질문에 답한다. 상위 운전자가 심야/할증 시간대에 집중적으로 운행하는지, 혹은 출퇴근 수요를 효율적으로 포착하는지를 비교한다. 시간대 선택이 매출 차이의 주요 원인인지, 아니면 같은 시간대에서도 OD 선택이 차이를 만드는지를 분리하여 분석한다.


In [ ]:
top_hour = pd.Series(0, index=range(24), dtype='int64')
bot_hour = pd.Series(0, index=range(24), dtype='int64')

cols = ['DRIVER_ID', 'RIDE_DTIME']

for i, chunk in enumerate(pd.read_csv(D012_PATH, chunksize=CHUNK_SIZE, dtype=DTYPE,
                                       usecols=cols)):
    chunk['DRIVER_ID'] = chunk['DRIVER_ID'].astype(str)
    chunk['hour'] = chunk['RIDE_DTIME'].astype(str).str[8:10].astype('int8')

    mask_top = chunk['DRIVER_ID'].isin(top_drivers)
    if mask_top.any():
        agg = chunk.loc[mask_top, 'hour'].value_counts()
        top_hour = top_hour.add(agg, fill_value=0)

    mask_bot = chunk['DRIVER_ID'].isin(bot_drivers)
    if mask_bot.any():
        agg = chunk.loc[mask_bot, 'hour'].value_counts()
        bot_hour = bot_hour.add(agg, fill_value=0)

    if (i + 1) % 50 == 0:
        print(f'  chunk {i+1} done')
    del chunk
    gc.collect()

top_hour = top_hour.astype('int64').sort_index()
bot_hour = bot_hour.astype('int64').sort_index()
mem_usage()

In [ ]:
# 비중으로 변환
top_pct = top_hour / top_hour.sum() * 100
bot_pct = bot_hour / bot_hour.sum() * 100

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(top_pct.index, top_pct.values, 'o-', color='#2196F3', linewidth=2, label='상위 10%')
ax.plot(bot_pct.index, bot_pct.values, 's--', color='#FF5722', linewidth=2, label='하위 10%')
ax.set_xlabel('시간대')
ax.set_ylabel('운행 비중 (%)')
ax.set_title('상위 vs 하위 운전자 시간대별 운행 비중')
ax.set_xticks(range(24))
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. 상위 vs 하위 운전자 지역 선호도 비교

승차 행정동 분포를 비교하여 상위 운전자가 선호하는 대기/영업 지역을 파악한다. 강남, 여의도 등 업무지구에서 대기하는 비율이 높은지, 혹은 역발상으로 경쟁이 적은 외곽 지역에서 효율적으로 운행하는지를 확인한다.


In [ ]:
top_region = pd.Series(dtype='int64')
bot_region = pd.Series(dtype='int64')

cols = ['DRIVER_ID', 'RIDE_A_CD']

for i, chunk in enumerate(pd.read_csv(D012_PATH, chunksize=CHUNK_SIZE, dtype=DTYPE,
                                       usecols=cols)):
    chunk['DRIVER_ID'] = chunk['DRIVER_ID'].astype(str)

    mask_top = chunk['DRIVER_ID'].isin(top_drivers)
    if mask_top.any():
        agg = chunk.loc[mask_top, 'RIDE_A_CD'].astype(str).value_counts()
        top_region = top_region.add(agg, fill_value=0)

    mask_bot = chunk['DRIVER_ID'].isin(bot_drivers)
    if mask_bot.any():
        agg = chunk.loc[mask_bot, 'RIDE_A_CD'].astype(str).value_counts()
        bot_region = bot_region.add(agg, fill_value=0)

    if (i + 1) % 50 == 0:
        print(f'  chunk {i+1} done')
    del chunk
    gc.collect()

top_region = top_region.astype('int64').sort_values(ascending=False)
bot_region = bot_region.astype('int64').sort_values(ascending=False)
mem_usage()

In [ ]:
# 상위 15개 지역 비교
top15 = top_region.head(15)
regions = top15.index.tolist()

top_pct_r = (top_region / top_region.sum() * 100).reindex(regions, fill_value=0)
bot_pct_r = (bot_region / bot_region.sum() * 100).reindex(regions, fill_value=0)

x = np.arange(len(regions))
w = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - w/2, top_pct_r.values, w, color='#2196F3', label='상위 10%')
ax.bar(x + w/2, bot_pct_r.values, w, color='#FF5722', label='하위 10%')
ax.set_xticks(x)
ax.set_xticklabels(regions, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('승차 비중 (%)')
ax.set_title('상위 vs 하위 운전자 승차 지역 Top 15 비교')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. 실차율 비교

실차율 = 운행거리 / (운행거리 + 빈차거리)

Cramer & Krueger (2016)가 핵심 효율성 지표로 사용한 실차율을 상위/하위 운전자 간 비교한다. 실차율 차이가 크다면, 상위 운전자의 골든 루트 전략이 빈차 시간 감소에 실질적으로 기여하고 있음을 의미한다.


In [ ]:
driver_ride_dist = pd.Series(dtype='float64')   # DRIVER_ID -> 총 RIDE_DIST
driver_vacant_dist = pd.Series(dtype='float64') # DRIVER_ID -> 총 VACNTV_DIST

cols = ['DRIVER_ID', 'RIDE_DIST', 'VACNTV_DIST']

for i, chunk in enumerate(pd.read_csv(D012_PATH, chunksize=CHUNK_SIZE, dtype=DTYPE,
                                       usecols=cols)):
    chunk['DRIVER_ID'] = chunk['DRIVER_ID'].astype(str)

    agg_ride = chunk.groupby('DRIVER_ID')['RIDE_DIST'].sum()
    agg_vacant = chunk.groupby('DRIVER_ID')['VACNTV_DIST'].sum()

    driver_ride_dist = driver_ride_dist.add(agg_ride, fill_value=0)
    driver_vacant_dist = driver_vacant_dist.add(agg_vacant, fill_value=0)

    if (i + 1) % 50 == 0:
        print(f'  chunk {i+1} done')
    del chunk, agg_ride, agg_vacant
    gc.collect()

total_dist = driver_ride_dist + driver_vacant_dist
occ_rate = (driver_ride_dist / total_dist.replace(0, np.nan) * 100).dropna()

top_occ = occ_rate[occ_rate.index.isin(top_drivers)]
bot_occ = occ_rate[occ_rate.index.isin(bot_drivers)]

print(f'상위 10% 평균 실차율: {top_occ.mean():.1f}%')
print(f'하위 10% 평균 실차율: {bot_occ.mean():.1f}%')

del driver_ride_dist, driver_vacant_dist, total_dist
gc.collect()
mem_usage()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
data_box = [top_occ.values, bot_occ.values]
bp = ax.boxplot(data_box, labels=['상위 10%', '하위 10%'], patch_artist=True,
                boxprops=dict(facecolor='white', edgecolor='black'),
                medianprops=dict(color='red', linewidth=2))
bp['boxes'][0].set(edgecolor='#2196F3', linewidth=2)
bp['boxes'][1].set(edgecolor='#FF5722', linewidth=2)
ax.set_ylabel('실차율 (%)')
ax.set_title('상위 vs 하위 운전자 실차율 비교')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. 요약

골든 루트 분석 결과를 종합한다. 핵심 발견: (1) 골든 루트 Top 10의 지리적/시간적 특성, (2) 상위 운전자의 행동 패턴(시간대, 지역, 실차율), (3) 하위 운전자가 개선할 수 있는 구체적 포인트.

실무 활용: 택시 회사나 플랫폼이 운전자 교육 시 골든 루트 정보를 제공하면, 하위 운전자의 생산성 향상이 기대된다. 나아가, 이 데이터는 AI 기반 배차 시스템의 추천 루트 알고리즘에 직접 반영 가능하다.


In [ ]:
print('=' * 60)
print('골든 루트 Top 10')
print('=' * 60)
print(golden_df.head(10)[['top_pct', 'bot_pct', 'gap']].to_string())
print()

print('=' * 60)
print('상위 운전자 행동 패턴 3가지')
print('=' * 60)
print('1. 골든 루트 집중: 하위에 없는 고빈도 OD를 반복 운행')
print('2. 시간대 전략: 특정 피크 시간대에 집중 배치')
print(f'3. 높은 실차율: 상위 {top_occ.mean():.1f}% vs 하위 {bot_occ.mean():.1f}% (빈차 시간 최소화)')
print()

print('=' * 60)
print('하위 운전자가 바꿔야 할 것')
print('=' * 60)
print('1. OD 패턴 전환: 골든 루트 지역으로 승차 지점 이동')
print('2. 시간대 재배치: 상위 운전자의 피크 시간대에 운행 집중')
print('3. 빈차 줄이기: 실차율 개선을 위한 대기 지점 최적화')
print('4. 수요 높은 지역 학습: 상위 운전자가 선호하는 승차 행정동 파악 후 이동')

## 10. [보강] 시계열 추세

In [ ]:
# === [시계열 보강] 일별 수요 추세 (7·30일 이동평균) ===
# 기존 분석과 독립적으로 일별 시계열을 다시 집계해 장기 추세를 확인한다.
import pandas as _pd, numpy as _np, matplotlib.pyplot as _plt
_daily = {}
for _ck in _pd.read_csv(D012_PATH if 'D012_PATH' in dir() else './DC_TBYXD012.csv',
                        usecols=['RIDE_DTIME'], dtype={'RIDE_DTIME': str}, chunksize=1_000_000):
    _d = _ck['RIDE_DTIME'].str[:8]
    _d = _d[_d.str.match(r'\d{8}')]
    for _k, _v in _d.groupby(_d).size().items():
        _daily[_k] = _daily.get(_k, 0) + _v
    del _ck
_ts = _pd.Series(_daily); _ts.index = _pd.to_datetime(_ts.index, format='%Y%m%d')
_ts = _ts.sort_index().asfreq('D').interpolate()
_ma7, _ma30 = _ts.rolling(7, center=True).mean(), _ts.rolling(30, center=True).mean()
fig, ax = _plt.subplots(figsize=(18, 5))
ax.plot(_ts.index, _ts.values, lw=0.3, alpha=0.4, color='gray', label='일별')
ax.plot(_ma7.index, _ma7.values, lw=1.2, color='steelblue', label='7일 이동평균')
ax.plot(_ma30.index, _ma30.values, lw=2, color='darkorange', label='30일 이동평균')
ax.set_title('일별 택시 수요 추세 (7·30일 이동평균)', fontweight='bold')
ax.set_xlabel('날짜'); ax.set_ylabel('일 건수'); ax.legend(); ax.grid(alpha=0.3)
_plt.tight_layout(); _plt.show()
print(f"기간 {_ts.index.min().date()} ~ {_ts.index.max().date()}, 일평균 {_ts.mean():,.0f}건")

---

## References

1. Cramer, J., & Krueger, A. B. (2016). Disruptive Change in the Taxi Business: The Case of Uber. *American Economic Review*, 106(5), 177-182.
2. Liu, L., Andris, C., & Ratti, C. (2013). Revealing travel patterns and city structure with taxi trip data. *Journal of Transport Geography*, 43, 78-90.
3. Tang, J., Liu, F., Wang, Y., & Wang, H. (2015). Uncovering urban human mobility from large scale taxi GPS data. *Physica A*, 438, 140-153.
